# [LAB] Multi-Agent 협업 실습

Agent 여럿이 일을 나눠 맡는 그래프를 만들고 실행해 봅니다. 주제는 모두 "조사해서 글로 쓰기"입니다. 누가 조사하고, 누가 쓰고, 누가 다음 차례를 정하는지를 실행 결과로 확인하는 것이 이 노트북의 목표입니다.

- **2장 Researcher와 Writer 나누기**: 조사하는 Agent와 글 쓰는 Agent를 한 그래프로 잇습니다. `# ✍️ CHANGE_HERE:` 줄에서 Researcher에게 준 지시를 바꿔 검색 횟수가 어떻게 달라지는지 봅니다.
- **3장 Supervisor가 다음 차례 정하기**: Supervisor가 researcher, writer, FINISH 중 하나를 골라 흐름을 이끕니다. `# ✍️ CHANGE_HERE:` 줄을 바꿔 Supervisor의 선택이 어떻게 달라지는지 봅니다.
- **4장 Critic 더하기** (선택): 초안의 치명적 오류를 찾는 Critic을 더합니다. Writer에게 준 지시를 바꿔 Critic의 판정이 어떻게 달라지는지 봅니다. 앞 장을 일찍 끝낸 분만 해 봅니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. 모든 장이 LLM을 여러 번 호출하므로 결과가 나오기까지 수십 초가 걸리고, 답은 실행할 때마다 조금씩 다릅니다.

**목차**

- 1. 실습 방법
- 2. Researcher와 Writer 나누기
  - 2.1 검색 도구
  - 2.2 공유 상태
  - 2.3 Researcher 노드
  - 2.4 도구를 쓸지 정하는 조건
  - 2.5 Writer 노드
  - 2.6 그래프 연결
  - 2.7 컴파일하고 실행하기
- 3. Supervisor가 다음 차례 정하기
  - 3.1 라우팅 결정 형식
  - 3.2 공유 상태
  - 3.3 Supervisor
  - 3.4 Researcher
  - 3.5 Writer
  - 3.6 라우팅 함수
  - 3.7 그래프 조립
  - 3.8 실행하기
- 4. Critic 더하기
  - 4.1 라우팅 결정 형식
  - 4.2 Critic 판정 형식
  - 4.3 공유 상태
  - 4.4 Supervisor
  - 4.5 Researcher
  - 4.6 Writer
  - 4.7 Critic
  - 4.8 라우팅 함수
  - 4.9 그래프 조립
  - 4.10 stream으로 실행하기
- 5. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. Researcher와 Writer 나누기

조사하는 Agent(Researcher)와 글 쓰는 Agent(Writer)를 나눠 한 그래프로 잇습니다. Researcher는 검색 도구 `search_web`을 쓸 수 있고, Writer는 도구 없이 조사 결과와 지금까지의 대화를 받아 글을 씁니다. 두 Agent는 같은 상태(State)를 함께 봅니다. 조사 결과는 `research_result`, 완성된 글은 `draft`에 담깁니다.

검색 도구는 실제로 검색하지 않고, 정해 둔 글(검색 결과 흉내)을 돌려줍니다.

그래프는 맨 아래 "컴파일하고 실행하기" 셀에서 처음 실행되므로, 위 셀을 차례로 실행해 두고 마지막 셀에서 결과를 봅니다.

**왜 배우나:** 역할을 나누면 Agent마다 필요한 도구와 지시만 갖게 됩니다. Researcher만 도구를 쓰고 Writer는 쓰지 않는 구조를 직접 만들어 보면, 역할 분리가 코드에서 어떤 모양인지 알 수 있습니다.

준비 코드입니다. 필요한 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 2.1 검색 도구

In [ ]:
# ── 도구 ──────────────────────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 주제를 조사한다."""
    # 실제 구현에서는 Tavily, SerpAPI 등을 사용한다.
    return (
        f'["{query}" 검색 결과]\n'
        f"- 해당 주제의 주요 동향: 최근 급격한 성장세를 보이고 있음\n"
        f"- 핵심 기술 요소: 트랜스포머 아키텍처, 대규모 데이터 학습\n"
        f"- 주요 플레이어: OpenAI, Anthropic, Google DeepMind\n"
        f"- 미래 전망: 향후 3년간 지속적인 발전이 예상됨"
    )

### 2.2 공유 상태

In [ ]:
# ── 상태 ──────────────────────────────────────────────────────────────────────
class State(TypedDict):
    messages: Annotated[list, add_messages]
    research_result: str  # Researcher가 수집한 정보
    draft: str  # Writer가 작성한 초안

### 2.3 Researcher 노드

**✍️ CHANGE_HERE: "철저히 조사하고"를 "딱 한 번만 검색해서 조사하고"로 바꾸면 검색 횟수가 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `철저히 조사하고` 자리에 복사해 붙여 넣기

```text
딱 한 번만 검색해서 조사하고
```

먼저 그대로 2.7까지 실행해 `Tool Message` 칸이 몇 개인지 봅니다. 바꾼 뒤 **이 셀부터 2.7 셀까지 차례로 다시 실행**하고(2.6에서 그래프를 다시 연결해야 반영됩니다), 확인했으면 원래대로 되돌립니다.

`researcher_node`는 Researcher가 한 번 생각하는 단계입니다. 시스템 메시지(역할)와 지금까지 쌓인 메시지를 도구가 연결된 모델 `researcher_llm`에 보내고, 받은 응답을 상태에 더합니다. 응답에 도구 호출 요청이 들어 있으면 다음 단계에서 검색 도구가 실행됩니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다.

In [ ]:
# ── Researcher Agent ──────────────────────────────────────────────────────────
# 역할: 주제를 조사해 research_result에 저장한다.

researcher_llm = llm.bind_tools([search_web])
researcher_tool_node = ToolNode([search_web])


def researcher_node(state: State) -> dict:
    # LLM의 역할 (페르소나) 정의
    system = SystemMessage(
        content=(
            "너는 전문 리서처야. "
            # ✍️ CHANGE_HERE: "철저히 조사하고"를 "딱 한 번만 검색해서 조사하고"로 바꾸면 검색 횟수가 어떻게 달라질까요?
            "search_web 도구를 사용해 주어진 주제를 철저히 조사하고, "
            "수집한 정보를 정리해서 제공해줘."
        )
    )
    response = researcher_llm.invoke([system] + state["messages"])
    return {"messages": [response]}


def researcher_tools_node(state: State) -> dict:
    # ToolNode 실행 후 research_result에 도구 결과를 저장
    tool_result = researcher_tool_node.invoke(state)
    last_tool_msg = tool_result["messages"][-1]
    return {
        "messages": tool_result["messages"],
        "research_result": last_tool_msg.content,
    }

<details><summary>무슨 일이 생기나</summary>

바꾸기 전에는 Researcher가 검색을 여러 번 합니다. `Tool Message` 칸이 8~20개쯤 나오고, 한 `Ai Message` 칸의 `Tool Calls:` 아래에 검색어 여러 개가 한꺼번에 들어 있기도 합니다. 셀이 끝나기까지 40~60초쯤 걸립니다.

바꾼 뒤에는 대개 `Tool Message` 칸이 1개만 나옵니다. 출력이 `Human Message` → `Ai Message`(도구 호출 1개) → `Tool Message` → `Ai Message`(Researcher의 정리) → `Ai Message`(Writer의 글)로 짧아지고, 20초쯤에 끝납니다. 이 실습의 검색 도구는 늘 같은 글을 돌려주므로, 검색을 많이 해도 Writer가 받는 조사 결과는 같습니다.

도구를 몇 번 쓸지는 그래프 코드가 아니라 Researcher에게 준 지시 한 줄이 정합니다. 도구 호출이 늘면 모델 호출과 시간·비용도 함께 늘어나므로, Agent마다 어디까지 할지를 지시에 적어 둡니다.

</details>

### 2.4 도구를 쓸지 정하는 조건

Researcher가 응답한 뒤 어디로 갈지 정하는 조건 함수입니다. 마지막 메시지에 도구 호출 요청(`tool_calls`)이 있으면 검색 도구 노드 `researcher_tools`로, 없으면 조사가 끝난 것이므로 `writer`로 보냅니다. 돌려주는 문자열은 그래프에 등록할 노드 이름과 똑같아야 합니다.

In [ ]:
# def researcher_condition(state: State) -> Literal["researcher_tools", "writer"]:  # 반환 타입 있음
def researcher_condition(state: State):  # 반환 타입 없음(지금)
    """Researcher가 도구 호출을 완료하면 writer로 넘어간다."""
    last = state["messages"][-1]

    if hasattr(last, "tool_calls") and last.tool_calls:
        return "researcher_tools"
    return "writer"

### 2.5 Writer 노드

In [ ]:
# ── Writer Agent ──────────────────────────────────────────────────────────────
# 역할: Researcher의 조사 결과를 바탕으로 완성된 글을 작성한다.


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content=(
            "너는 전문 작가야. "
            "제공된 리서치 결과를 바탕으로 독자가 이해하기 쉬운 글을 작성해줘. "
            "서론, 본론, 결론 구조를 갖춰줘."
        )
    )
    context = f"[리서치 결과]\n{state.get('research_result', '')}"
    response = llm.invoke(
        [
            system,
            HumanMessage(content=context),
            *state["messages"],
        ]
    )
    return {"messages": [response], "draft": response.content}

### 2.6 그래프 연결

이 셀은 노드 세 개(`researcher`, `researcher_tools`, `writer`)를 등록하고 흐름을 잇습니다. 시작하면 researcher로 가고, researcher 다음은 위에서 만든 조건 함수가 정하고, 검색 도구를 실행한 뒤에는 다시 researcher로 돌아오고, writer가 끝나면 그래프도 끝납니다.

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("researcher", researcher_node)
builder.add_node("researcher_tools", researcher_tools_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "researcher")
builder.add_conditional_edges("researcher", researcher_condition)
builder.add_edge("researcher_tools", "researcher")
builder.add_edge("writer", END);

### 2.7 컴파일하고 실행하기

연결을 마친 그래프를 실행할 수 있는 앱으로 만듭니다. 그 아래 코드는 요청 한 문장을 넣어 그래프를 실행하고, 그래프에 쌓인 메시지를 차례로 출력합니다. 앞의 셀(2.1~2.6)을 모두 실행한 뒤 이 셀을 실행합니다.

In [ ]:
app = builder.compile()

researcher_writer_result = app.invoke(
    {
        "messages": [
            HumanMessage(content="AI Agent 기술의 현황과 전망을 조사해서 글로 써줘.")
        ],
        "research_result": "",
        "draft": "",
    }
)
results = researcher_writer_result["messages"]
for m in results:
    m.pretty_print()

# 그래프 구조를 그림으로 출력
from draw import draw

draw(app)

**볼 지점**
- `Tool Message` 칸 하나가 검색 한 번의 결과입니다. 칸 안의 다섯 줄이 검색 도구가 돌려준 조사 결과이고, 마지막 검색 결과가 State의 `research_result`에도 담깁니다. 첫 줄 따옴표 안이 Researcher가 만든 검색어이고, 그 위 `Ai Message` 칸의 `Tool Calls:`에도 보입니다.
- Researcher는 한 번에 여러 검색어로 도구를 부르기도 합니다. 그러면 한 `Ai Message` 칸의 `Tool Calls:` 아래에 검색어가 여러 개 있고, `Tool Message` 칸이 그 수만큼 이어 나옵니다. 그다음 Researcher가 결과를 보고 검색을 더 하거나 조사를 마칩니다.
- 흐름은 researcher → researcher_tools → researcher(검색이 끝날 때까지 되풀이) → writer입니다. 끝에서 두 번째 `Ai Message` 칸(도구 호출 없음)이 Researcher가 검색 결과를 보고 정리한 답이고, 그다음 조건 함수가 writer로 보냅니다.
- 맨 아래 `Ai Message` 칸이 Writer의 글입니다. 같은 글이 State의 `draft`에도 담깁니다. 조사 결과의 "주요 플레이어" 이름이 글에 없을 수도 있습니다. Writer는 조사 결과를 바탕으로 하되, 모델이 아는 내용도 섞어 씁니다.

## 3. Supervisor가 다음 차례 정하기

이번에는 Supervisor가 다음에 누가 일할지 정합니다. Supervisor는 매번 researcher, writer, FINISH 중 하나를 고르고, 고른 이유를 함께 남깁니다. 하위 Agent는 일을 마치면 다시 Supervisor에게 돌아옵니다. 이 실습의 Supervisor는 다음 차례만 고르고 결과를 종합하지는 않습니다. 최종 글은 Writer가 쓴 초안입니다.

Supervisor의 선택은 `RouteDecision`이라는 정해진 형식으로 받습니다. `next`에는 세 값 중 하나만 들어갈 수 있어서, 모델이 엉뚱한 이름을 만들어 내도 그래프가 길을 잃지 않습니다. 이번 장의 Researcher는 도구 없이 모델이 알고 있는 내용으로 조사합니다.

코드는 완성되어 있습니다. 먼저 그대로 끝까지 실행해 출력을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행해 결과를 비교합니다.

**왜 배우나:** 앞 장은 다음 차례를 조건 함수(코드)가 정했습니다. 이번 장은 모델이 정합니다. 모델에게 맡긴 결정이 어떻게 흔들릴 수 있는지 보고, 그 원인을 코드에서 찾는 연습을 합니다.

이 장의 준비 코드입니다. 앞 장과 같은 모델을 쓰고, 정해진 형식으로 답을 받는 데 필요한 `BaseModel`을 더 불러옵니다. 실행만 합니다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 3.1 라우팅 결정 형식

In [ ]:
# ── Supervisor가 내리는 라우팅 결정 스키마 ─────────────────────────────────────
class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "FINISH"]
    reason: str


# 수퍼바이저를 위한 LLM을 하나 정의
supervisor_llm = llm.with_structured_output(RouteDecision)

### 3.2 공유 상태

In [ ]:
# ── 공유 State ────────────────────────────────────────────────────────────────


class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str  # Supervisor가 결정한 다음 에이전트
    research_result: str
    draft: str

### 3.3 Supervisor

In [ ]:
# ── Supervisor ────────────────────────────────────────────────────────────────
# 사용자의 요청과 현재 진행 상황을 보고 어떤 에이전트를 호출할지 결정한다.

SUPERVISOR_PROMPT = """너는 멀티 에이전트 시스템의 Supervisor야.
다음 에이전트 중 하나를 선택해 작업을 지시하거나 완료를 선언해줘.

- researcher: 웹 검색으로 정보를 수집해야 할 때
- writer: 수집된 정보를 바탕으로 글을 작성해야 할 때
- FINISH: 모든 작업이 완료됐을 때

현재 상황:
- 리서치 완료 여부: {research_done}
- 초안 작성 여부: {draft_done}
"""


def supervisor_node(state: State) -> dict:
    prompt = SUPERVISOR_PROMPT.format(
        research_done="완료" if state.get("research_result") else "미완료",
        draft_done="완료" if state.get("draft") else "미완료",
    )
    decision = supervisor_llm.invoke(
        [
            SystemMessage(content=prompt),
            *state["messages"],
        ]
    )
    return {
        "next": decision.next,
        "messages": [
            AIMessage(content=f"[Supervisor → {decision.next}] {decision.reason}")
        ],
    }

### 3.4 Researcher

**✍️ CHANGE_HERE: 괄호 안 끝에 "핵심 사실만 5줄 이내 목록으로 답해줘. 글은 쓰지 마."를 더하면 Supervisor가 고르는 차례가 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `답해줘.)` 자리에 복사해 붙여 넣기

```text
답해줘. 핵심 사실만 5줄 이내 목록으로 답해줘. 글은 쓰지 마.)
```

먼저 그대로 3.8까지 실행해 결과를 봅니다. 바꾼 뒤 **이 셀부터 3.8 셀까지 차례로 다시 실행**하고(3.7에서 그래프를 다시 조립해야 반영됩니다), 확인했으면 원래대로 되돌립니다.

In [ ]:
# ── Researcher ────────────────────────────────────────────────────────────────


def researcher_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 리서처야. 주제를 조사해서 핵심 정보를 정리해줘."
    )
    topic = state["messages"][0].content  # 사용자의 원래 요청
    result = llm.invoke(
        [
            system,
            HumanMessage(
                # ✍️ CHANGE_HERE: 괄호 안 끝에 "핵심 사실만 5줄 이내 목록으로 답해줘. 글은 쓰지 마."를 더하면 Supervisor가 고르는 차례가 어떻게 달라질까요?
                content=f"다음 주제를 조사해줘: {topic}\n(실제 검색 대신 알고 있는 내용으로 답해줘.)"
            ),
        ]
    )
    return {
        "research_result": result.content,
        "messages": [AIMessage(content=f"[Researcher 완료]\n{result.content}")],
    }

<details><summary>무슨 일이 생기나</summary>

바꾸기 전에는 `[Researcher 완료]` 아래에 `#` 제목이 달린 완성된 블로그 글(100줄 안팎)이 나옵니다. Supervisor가 이 글을 초안으로 보고 writer를 건너뛰어 곧바로 `→ FINISH`를 고르는 일이 자주 있습니다. 이때 `[Supervisor → FINISH]` 줄의 이유에 "블로그 초안이 작성되었습니다" 같은 말이 보이고, "최종 초안" 아래는 비어 있습니다. `draft`를 채우는 것은 Writer뿐이기 때문입니다.

바꾼 뒤에는 `[Researcher 완료]` 아래가 `-`로 시작하는 5줄 목록이 됩니다. 흐름은 대개 `→ researcher`, `→ writer`, `→ FINISH` 순서로 가고, "최종 초안" 아래에 Writer의 글이 나옵니다. Researcher가 짧게 답하므로 실행 시간도 20~45초에서 15초쯤으로 줄어듭니다.

Supervisor는 메시지에 쌓인 결과의 모양을 보고 다음 차례를 정합니다. Researcher가 Writer의 일까지 해 버리면 역할 경계가 흐려지고, Supervisor도 판단을 잘못합니다. Agent마다 돌려줄 출력의 형식을 정해 두는 이유입니다.

</details>

### 3.5 Writer

In [ ]:
# ── Writer ────────────────────────────────────────────────────────────────────


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 작가야. 리서치 결과를 바탕으로 완성도 높은 글을 써줘."
    )
    result = llm.invoke(
        [
            system,
            HumanMessage(
                content=f"다음 리서치 결과로 글을 써줘:\n\n{state['research_result']}"
            ),
        ]
    )
    return {
        "draft": result.content,
        "messages": [AIMessage(content=f"[Writer 완료]\n{result.content}")],
    }

### 3.6 라우팅 함수

In [ ]:
# ── 라우팅 함수 ───────────────────────────────────────────────────────────────
def route(state: State) -> Literal["researcher", "writer", "__end__"]:
    return state["next"] if state["next"] != "FINISH" else END

### 3.7 그래프 조립

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route)
builder.add_edge("researcher", "supervisor")  # 완료 후 Supervisor에게 보고
builder.add_edge("writer", "supervisor")  # 완료 후 Supervisor에게 보고

app = builder.compile()

### 3.8 실행하기

In [ ]:
supervisor_result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="LangGraph Multi-Agent 시스템에 대한 블로그 포스트를 써줘."
            )
        ],
        "next": "",
        "research_result": "",
        "draft": "",
    }
)

print("=== Supervisor 의사결정 흐름 ===")
results = supervisor_result["messages"]
for m in results:
    m.pretty_print()

print("\n=== 최종 초안 ===")
print(supervisor_result["draft"])

**볼 지점**
- "Supervisor 의사결정 흐름"의 `[Supervisor → 이름]` 줄이 Supervisor가 고른 다음 차례와 이유입니다. 화살표 뒤 이름은 researcher, writer, FINISH 셋 중 하나만 나옵니다.
- 바꾸기 전에는 `→ writer` 없이 `→ FINISH`로 끝나고 "최종 초안" 아래가 비어 있는 일이 자주 있습니다. 이때 `[Researcher 완료]` 아래가 조사 결과가 아니라 `#` 제목이 달린 완성된 글인지 봅니다.
- CHANGE_HERE 줄을 바꾼 뒤에는 `[Researcher 완료]` 아래의 모양과 Supervisor가 고른 차례를 바꾸기 전과 비교합니다. 결과는 모델이 정하므로 실행할 때마다 다를 수 있습니다.
- 셀이 몇 분이 지나도 끝나지 않으면 같은 Agent를 되풀이하고 있는 것입니다. 셀 실행을 중지(■)하고 바꾼 줄을 다시 봅니다.

## 4. Critic 더하기

이 장은 선택입니다. 앞 장을 일찍 끝낸 분만 해 봅니다.

앞 장의 그래프에 Critic을 더합니다. Critic은 결과물을 검토하는 Reviewer 역할의 Agent입니다. Critic은 초안에서 사실 오류, 논리 모순, 핵심 요구 누락 같은 **치명적 오류만** 찾습니다. 오류를 찾으면 Supervisor가 writer에게 수정을 맡깁니다. 고친 초안을 Critic에게 다시 보낼지도 Supervisor가 정합니다. Writer는 처음 쓸 때 일부러 치명적인 오류를 섞으라는 지시를 받습니다. Critic이 오류를 잡아내는지 보려는 장치입니다. Researcher 지시에는 3.4에서 더해 본 형식 지시("핵심 사실만 5줄 이내 목록으로 답해줘. 글은 쓰지 마.")를 미리 넣었습니다.

실행은 `stream`으로 합니다. 노드가 하나 끝날 때마다 `[노드 이름]`과 결과의 앞 100자를 출력합니다.

코드는 완성되어 있습니다. 먼저 그대로 실행하고, `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행해 흐름을 비교합니다.

이 장의 준비 코드도 앞 장과 거의 같습니다. 펼치지 않고 실행만 합니다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 4.1 라우팅 결정 형식

In [ ]:
# ── Supervisor 가 내리는 라우팅 결정 스키마 ─────────────────────────────────────


class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "critic", "FINISH"]
    reason: str


supervisor_llm = llm.with_structured_output(RouteDecision)

### 4.2 Critic 판정 형식

In [ ]:
# ── Critic 이 분석하는 치명적 오류 스키마 ──────────────────────────────────────


class CriticFeedback(BaseModel):
    has_fatal_error: bool
    error_type: str  # 'factual_error', 'logical_error', 'missing_key_point', 'none'
    description: str
    suggestion: str


critic_llm = llm.with_structured_output(CriticFeedback)

### 4.3 공유 상태

In [ ]:
# ── 공유 State ────────────────────────────────────────────────────────────────


class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str  # Supervisor 가 결정한 다음 에이전트
    research_result: str
    draft: str
    critic_feedback: str  # Critic 의 개선점 피드백
    revision_count: int  # 수정 횟수 (무한 루프 방지)

### 4.4 Supervisor

In [ ]:
# ── Supervisor ────────────────────────────────────────────────────────────────
# 사용자의 요청과 현재 진행 상황을 보고 어떤 에이전트를 호출할지 결정한다.

SUPERVISOR_PROMPT = """너는 멀티 에이전트 시스템의 Supervisor 야.
다음 에이전트 중 하나를 선택해 작업을 지시하거나 완료를 선언해줘.

- researcher: 웹 검색으로 정보를 수집해야 할 때
- writer: 수집된 정보를 바탕으로 글을 작성해야 할 때
- critic: 작성된 초안의 치명적 오류를 분석해야 할 때
- FINISH: 모든 작업이 완료됐을 때

현재 상황:
- 리서치 완료 여부: {research_done}
- 초안 작성 여부: {draft_done}
- Critic 피드백 존재: {critic_done}
- 현재 수정 횟수: {revision_count}

주의:
- Critic 이 치명적 오류를 찾았다면 writer 를 호출해서 수정하게 해줘.
- 수정 횟수가 4 회를 넘었다면 FINISH 를 선택해줘.
"""


def supervisor_node(state: State) -> dict:
    prompt = SUPERVISOR_PROMPT.format(
        research_done="완료" if state.get("research_result") else "미완료",
        draft_done="완료" if state.get("draft") else "미완료",
        critic_done="있음" if state.get("critic_feedback") else "없음",
        revision_count=state.get("revision_count", 0),
    )
    decision = supervisor_llm.invoke(
        [
            SystemMessage(content=prompt),
            *state["messages"],
        ]
    )
    return {
        "next": decision.next,
        "messages": [
            AIMessage(content=f"[Supervisor → {decision.next}] {decision.reason}")
        ],
    }

### 4.5 Researcher

In [ ]:
# ── Researcher ────────────────────────────────────────────────────────────────


def researcher_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 리서처야. 주제를 조사해서 핵심 정보를 정리해줘."
    )
    topic = state["messages"][0].content  # 사용자의 원래 요청
    result = llm.invoke(
        [
            system,
            HumanMessage(
                # 3.4에서 더해 본 형식 지시를 미리 넣었습니다
                content=f"다음 주제를 조사해줘: {topic}\n(실제 검색 대신 알고 있는 내용으로 답해줘. 핵심 사실만 5줄 이내 목록으로 답해줘. 글은 쓰지 마.)"
            ),
        ]
    )
    return {
        "research_result": result.content,
        "messages": [AIMessage(content=f"[Researcher 완료]\n{result.content}")],
    }

### 4.6 Writer

**✍️ CHANGE_HERE: 끝의 ". 단 의도적으로 치명적인 오류를 섞어 넣어줘"를 지우면 Critic의 판정과 수정 횟수가 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄에서 `. 단 의도적으로 치명적인 오류를 섞어 넣어줘` 지우기

먼저 그대로 4.10까지 실행해 흐름을 봅니다. 바꾼 뒤 **이 셀부터 4.10 셀까지 차례로 다시 실행**하고(4.9에서 그래프를 다시 조립해야 반영됩니다), 확인했으면 지운 글을 다시 넣어 둡니다.

In [ ]:
# ── Writer ────────────────────────────────────────────────────────────────────


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content="너는 작가야. 리서치 결과를 바탕으로 완성도 높은 글을 써줘."
    )

    # Critic 피드백이 있다면 이를 반영하여 수정
    if state.get("critic_feedback"):
        prompt = f"""
        다음 리서치 결과로 글을 수정해줘.

        [기존 초안]
        {state["draft"]}

        [Critic 피드백 - 치명적 오류]
        {state["critic_feedback"]}

        위 피드백을 반영하여 치명적 오류를 수정한 새로운 버전을 작성해줘.
        """
    else:
        # ✍️ CHANGE_HERE: 끝의 ". 단 의도적으로 치명적인 오류를 섞어 넣어줘"를 지우면 Critic의 판정과 수정 횟수가 어떻게 달라질까요?
        prompt = f"다음 리서치 결과로 글을 써줘:\n\n{state['research_result']}. 단 의도적으로 치명적인 오류를 섞어 넣어줘"

    result = llm.invoke(
        [
            system,
            HumanMessage(content=prompt),
        ]
    )
    return {
        "draft": result.content,
        "critic_feedback": "",
        "revision_count": state.get("revision_count", 0) + 1,
        "messages": [
            AIMessage(
                content=f"[Writer 완료 - 수정 {state.get('revision_count', 0) + 1}번째]\n{result.content}"
            )
        ],
    }

<details><summary>무슨 일이 생기나</summary>

바꾸기 전에도 Writer가 틀린 문장을 섞지 않고 바르게 쓰는 일이 많아, 실행마다 결과가 다릅니다. 틀린 문장을 섞은 때(열 번 중 서너 번)는 Critic이 `[치명적 오류 발견: …]`으로 잡아내고(예: 개발한 회사나 동작 방식을 틀리게 쓴 문장), Supervisor가 writer에게 다시 맡겨 `[Writer 완료 - 수정 2번째]`가 나온 뒤 끝납니다. 고친 초안은 Critic을 다시 거치지 않고 끝나는 일이 많습니다. 틀린 문장이 없는 때는 바꾼 뒤와 같은 결과입니다.

바꾼 뒤에는 Critic이 대개 `[치명적 오류 없음]`이라고 답하고, Supervisor가 곧바로 FINISH를 골라 `수정 1번째`에서 끝납니다. 거치는 노드가 줄어 실행도 25초쯤에서 18초쯤으로 짧아집니다. 바꾸기 전 결과도 `[치명적 오류 없음]`이었다면 차이가 보이지 않으니, 바꾸기 전 셀을 두세 번 더 실행해 봅니다.

Critic(Reviewer)은 고칠 것을 찾았을 때만 writer를 한 번 더 부르게 만듭니다. 검토와 수정을 되풀이할수록 모델 호출이 늘어 시간과 비용이 드니, 검토 단계는 그만큼 효과가 있는지 보고 둡니다.

</details>

### 4.7 Critic

In [ ]:
# ── Critic ────────────────────────────────────────────────────────────────────

CRITIC_PROMPT = """너는 치명적 오류만 분석하는 Critic 이야.
다음 기준에서 **치명적 오류**만 찾아내:
1. 사실적 오류 (Fact-checking 실패)
2. 논리적 모순 (앞뒤 내용이 맞지 않음)
3. 핵심 요구사항 누락 (사용자 요청을 충족하지 못함)

사소한 문체, 오타, 스타일 문제는 무시해.
치명적 오류가 없다면 has_fatal_error=False 로 응답해.
"""


def critic_node(state: State) -> dict:
    feedback = critic_llm.invoke(
        [
            SystemMessage(content=CRITIC_PROMPT),
            HumanMessage(content=f"다음 초안을 분석해줘:\n\n{state['draft']}"),
        ]
    )

    if feedback.has_fatal_error:
        critic_msg = f"[치명적 오류 발견: {feedback.error_type}]\n{feedback.description}\n\n[개선 제안]\n{feedback.suggestion}"
    else:
        critic_msg = "[치명적 오류 없음]\n초안이 양호합니다."

    return {
        "critic_feedback": critic_msg if feedback.has_fatal_error else "",
        "messages": [AIMessage(content=f"[Critic 완료]\n{critic_msg}")],
    }

### 4.8 라우팅 함수

In [ ]:
# ── 라우팅 함수 ───────────────────────────────────────────────────────────────


def route(state: State) -> Literal["researcher", "writer", "critic", "__end__"]:
    return state["next"] if state["next"] != "FINISH" else END

### 4.9 그래프 조립

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)
builder.add_node("critic", critic_node)
builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route)

# 각 노드 완료 후 Supervisor 에게 보고
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")
builder.add_edge("critic", "supervisor")
app = builder.compile()

### 4.10 stream으로 실행하기

In [ ]:
# ── stream 형식으로 실행 (각 노드별 실시간 진행 확인) ─────────────────────────

print("=== Agent 실행 시작 ===")

for chunk in app.stream(
    {
        "messages": [
            HumanMessage(
                content="LangGraph Multi-Agent 시스템에 대한 블로그 포스트를 써줘."
            )
        ],
        "next": "",
        "research_result": "",
        "draft": "",
        "critic_feedback": "",
        "revision_count": 0,
    }
):
    for node_name in chunk.keys():
        print(f"========= [{node_name}] =========")
        message_type = chunk[node_name]["messages"][0].__class__.__name__
        if message_type == "AIMessage":
            print("AIMessage")
        elif message_type == "ToolMessage":
            print("ToolMessage")
        print(chunk[node_name]["messages"][0].content[:100])

**볼 지점**
- `=========` 사이의 이름이 방금 끝난 노드입니다. supervisor가 다른 노드 사이사이에 끼어 다음 차례를 정합니다.
- `[researcher]` 블록이 짧은 목록으로 시작하고, 그다음 `[writer]`가 나오는지 봅니다. Researcher 지시에 출력 형식을 미리 넣어 두었기 때문입니다.
- `[critic]` 블록의 판정(`치명적 오류 발견` 또는 `치명적 오류 없음`)과 `[writer]` 블록의 "수정 N번째"가 몇까지 올라가는지 봅니다.
- writer와 critic이 다섯 번 넘게 오가며 끝나지 않으면 셀 실행을 중지(■)합니다.

## 5. 정리

Agent를 나누고, 다음 차례를 정하는 방법을 세 가지로 보았습니다.

| 장 | 다음 차례를 정하는 것 | 확인한 것 |
|---|---|---|
| 2. Researcher와 Writer 나누기 | 조건 함수(코드)가 `tool_calls`를 보고 정함 | Researcher만 도구를 쓰고, 조사 결과를 Writer가 받아 글을 씀. 검색 횟수는 Researcher 지시 한 줄로 달라짐 |
| 3. Supervisor가 다음 차례 정하기 | Supervisor 모델이 정해진 형식(`next`)으로 고름 | 모델의 선택은 흔들릴 수 있고, Researcher의 출력 형식을 정하면 흐름이 안정됨 |
| 4. Critic 더하기 (선택) | Supervisor 모델이 Critic 판정을 보고 고름 | 오류가 없으면 수정 없이 끝나고, 반복을 끝내는 조건은 코드가 아니라 프롬프트에만 있음 |

역할을 나눌 때는 Agent마다 받을 입력과 돌려줄 출력의 형식을 정해야 합니다. 형식이 흐리면 Supervisor처럼 다음 차례를 정하는 쪽도 판단을 잘못할 수 있습니다.

값을 바꿔 다시 실행하고, 달라진 결과를 "무슨 일이 생기나"와 비교했습니다.